# EDA and Isolation Forest Baseline

Load EDP SCADA, verify gearbox channels and failure events, train per-turbine Isolation Forest on healthy history, and plot anomaly scores against known failure dates.

For the full benchmark (pure ML + physics-residual hybrid), run `python scripts/run_all_ml_baselines.py` — see [README.md](../README.md) and the [blog post](../docs/BLOG.md).

**Prerequisites:** Raw EDP files in `data/raw/edp/` — see [data/README.md](../data/README.md).

**Docs:** [DATA.md](../docs/DATA.md) | [EVALUATION.md](../docs/EVALUATION.md)

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from wind_turbine_anomaly.config import (
    DATA_RAW,
    DEFAULT_BUFFER_DAYS,
    DEFAULT_CONTAMINATION,
    DEFAULT_HORIZON_DAYS,
    DEFAULT_THRESHOLD_PERCENTILE,
    FEATURE_COLUMNS,
    RESULTS_DIR,
)
from wind_turbine_anomaly.data.clean import (
    clean_turbine_df,
    get_failure_for_turbine,
    healthy_training_mask,
)
from wind_turbine_anomaly.data.load_edp import load_edp_dataset
from wind_turbine_anomaly.eval.protocol import evaluate_turbine, results_to_dict
from wind_turbine_anomaly.models.isolation_forest import (
    fit_isolation_forest,
    save_scores,
    threshold_from_training,
)

RAW_DIR = DATA_RAW
print(f"Loading from {RAW_DIR}")

In [ ]:
turbines, gearbox_failures = load_edp_dataset(RAW_DIR)

print(f"Turbines: {sorted(turbines.keys())}")
print(f"Gearbox failures in log: {len(gearbox_failures)}")
for g in gearbox_failures:
    print(f"  {g.turbine_id}  {g.timestamp}  {g.remarks}")

sample = next(iter(turbines.values()))
missing = [c for c in FEATURE_COLUMNS if c not in sample.columns]
assert not missing, f"Missing channels: {missing}"
print("All gearbox-relevant channels present.")

In [ ]:
out_dir = RESULTS_DIR / "isolation_forest"
out_dir.mkdir(parents=True, exist_ok=True)

eval_results = []
pipelines = {}
thresholds = {}

for turbine_id, raw_df in sorted(turbines.items()):
    df = clean_turbine_df(raw_df)
    failure = get_failure_for_turbine(turbine_id, gearbox_failures)
    train_mask = healthy_training_mask(df.index, failure, DEFAULT_BUFFER_DAYS)
    train_df = df.loc[train_mask]

    if len(train_df) < 100:
        print(f"SKIP {turbine_id}: only {len(train_df)} training rows")
        continue

    pipeline = fit_isolation_forest(
        train_df, FEATURE_COLUMNS, contamination=DEFAULT_CONTAMINATION
    )
    train_scores = pipeline.score(train_df)
    threshold = threshold_from_training(
        train_scores, percentile=DEFAULT_THRESHOLD_PERCENTILE
    )

    score_df = df.loc[~train_mask] if (~train_mask).any() else df
    scores = pipeline.score(score_df)
    save_scores(scores, out_dir / f"{turbine_id}_scores.parquet")

    pipelines[turbine_id] = pipeline
    thresholds[turbine_id] = threshold

    result = evaluate_turbine(
        turbine_id,
        scores,
        threshold=threshold,
        failure=failure,
        horizon_days=DEFAULT_HORIZON_DAYS,
        score_start=train_df.index[-1],
    )
    eval_results.append(result)
    print(
        f"{turbine_id}: lead={result.lead_time_days}, "
        f"false_alarms={result.false_alarm_episodes}"
    )

metrics = results_to_dict(eval_results, DEFAULT_HORIZON_DAYS)
metrics_path = out_dir / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved {metrics_path}")

In [ ]:
failure_targets = ["T01", "T06"]

fig, axes = plt.subplots(len(failure_targets), 1, figsize=(14, 4 * len(failure_targets)), sharex=False)
if len(failure_targets) == 1:
    axes = [axes]

for ax, tid in zip(axes, failure_targets):
    if tid not in turbines:
        ax.set_title(f"{tid} — not in dataset")
        continue

    df = clean_turbine_df(turbines[tid])
    failure = get_failure_for_turbine(tid, gearbox_failures)
    train_mask = healthy_training_mask(df.index, failure, DEFAULT_BUFFER_DAYS)
    train_df = df.loc[train_mask]

    if tid not in pipelines:
        ax.set_title(f"{tid} — skipped (insufficient data)")
        continue

    score_df = df.loc[~train_mask] if (~train_mask).any() else df
    scores = pipelines[tid].score(score_df)
    threshold = thresholds[tid]

    ax.plot(scores.index, scores.values, lw=0.5, label="Anomaly score")
    ax.axhline(threshold, color="orange", ls="--", label=f"Threshold ({DEFAULT_THRESHOLD_PERCENTILE}th pct)")

    if failure:
        ft = pd.Timestamp(failure.timestamp, tz="UTC")
        ax.axvline(ft, color="red", ls="-", label=f"Failure: {failure.remarks}")

    ax.set_title(f"{tid} — Isolation Forest anomaly score")
    ax.set_ylabel("Score (higher = more anomalous)")
    ax.legend(loc="upper left")

plt.xlabel("Time (UTC)")
plt.tight_layout()
fig.savefig(out_dir / "score_vs_failure.png", dpi=150)
plt.show()

In [ ]:
pd.DataFrame(metrics["turbines"])